In [1]:
%load_ext autoreload
%autoreload 2

In [11]:
import numpy as np
import copy
from tqdm import tqdm
from pathlib import Path
import networkx as nx

from src.load_data import (
    read_graph_transport_networks_tntp,
    read_traffic_mat_transport_networks_tntp,
    read_graph_sndlib_xml,
    read_traffic_mat_sndlib_xml,
    scale_graph_bandwidth_and_cost
)

from src.shortest_paths_gt import get_graph_props

from src.models import BeckmannModel, TelecomModel
from src.algs import cyclic, ustm, frank_wolfe, N_conjugate_frank_wolfe
from src.salim import SaddleOracle, combined_salim
from src.saddle_ta import salim_ta, chambolle_pock_ta
from src.approx import seq_quad, seq_quad_chp
from src.path_based import pb_gradproj_ta

import matplotlib.pyplot as plt
from matplotlib.ticker import LogLocator

import pandas as pd

import json


plt.rcParams.update({'font.size': 13})
%config InlineBackend.figure_format = 'retina'

%matplotlib inline

In [20]:
networks_path = Path("./TransportationNetworks")
nets = [
    dict(folder="SiouxFalls",
        net_name = "SiouxFalls_net",
        traffic_mat_name = "SiouxFalls_trips"),
    dict(folder = "Anaheim",
        net_name = "Anaheim_net",
        traffic_mat_name = "Anaheim_trips"),
    dict(folder = "Terrassa-Asymmetric",
        net_name = "Terrassa-Asym_net",
        traffic_mat_name = "Terrassa-Asym_trips"),
        
    dict(folder = "Berlin-Mitte-Center",
        net_name = "berlin-mitte-center_net",
        traffic_mat_name = "berlin-mitte-center_trips"),
        
    dict(folder = "Berlin-Tiergarten",
        net_name = "berlin-tiergarten_net",
        traffic_mat_name = "berlin-tiergarten_trips"),
        
    dict(folder = "Eastern-Massachusetts",
        net_name = "EMA_net",
        traffic_mat_name = "EMA_trips"),
]

res = []
for d in nets:
    (folder, net_name, traffic_mat_name) = d.values()

    net_file = networks_path / folder / f"{net_name}.tntp"
    traffic_mat_file = networks_path / folder / f"{traffic_mat_name}.tntp"
    graph, metadata = read_graph_transport_networks_tntp(net_file)
    # graph = scale_graph_bandwidth_and_cost(graph)
    correspondences = read_traffic_mat_transport_networks_tntp(traffic_mat_file, metadata)
    n = graph.number_of_nodes()
    
    traffic_mat = correspondences.traffic_mat.copy()
    departures, arrivals = traffic_mat.sum(axis=1), traffic_mat.sum(axis=0)
    l, w = departures, arrivals
    
    mu_bm = None
    beckmann_model = BeckmannModel(
        graph,
        copy.deepcopy(correspondences),
        mu_bm,
    )
    
    _, _, logs, _ = frank_wolfe(
        beckmann_model,
        eps_abs=None,
        max_iter=10,
        stop_by_crit=False,
        linesearch=True,
        log_max_diff=None,
        log_period=1,
        solution_flows=None,
        benchmark_cugraph_sssp=True,
    )
    cpu_runtimes = logs[-2]
    gpu_runtimes = logs[-1]
    res.append(dict(net=folder, cpu_runtimes=cpu_runtimes, gpu_runtimes=gpu_runtimes))

with open("cugraph_vs_cpu.json", "w") as fp:
    json.dump(res, fp)


metadata["can_pass_through_zones"]=True


100%|██████████| 10/10 [00:01<00:00,  6.49it/s]


metadata["can_pass_through_zones"]=False


100%|██████████| 10/10 [00:05<00:00,  1.71it/s]


metadata["can_pass_through_zones"]=False


...

100%|██████████| 10/10 [00:06<00:00,  1.58it/s]


metadata["can_pass_through_zones"]=False


100%|██████████| 10/10 [00:04<00:00,  2.21it/s]


metadata["can_pass_through_zones"]=True


100%|██████████| 10/10 [00:05<00:00,  1.74it/s]


In [22]:
with open("cugraph_vs_cpu.json", "r") as fp:
    res = json.load(fp)
    
df_list = []
for r in res:
    cpu_runtimes = r["cpu_runtimes"]
    gpu_runtimes = r["gpu_runtimes"]
    cpu_mean = np.mean(cpu_runtimes)
    gpu_mean = np.mean(gpu_runtimes)
    cpu_std = np.std(cpu_runtimes)
    gpu_std = np.std(gpu_runtimes)
    df_list.append(dict(net=r["net"], cpu_mean=cpu_mean, cpu_std=cpu_std, gpu_mean=gpu_mean, gpu_std=gpu_std))

pd.DataFrame(df_list)

,net,cpu_mean,cpu_std,gpu_mean,gpu_std
0,SiouxFalls,0.003745,0.000437,0.147165,0.013532
1,Anaheim,0.011138,0.002994,0.569404,0.018448
2,Terrassa-Asymmetric,0.027640,0.001376,2.514455,0.015969
3,Berlin-Mitte-Center,0.009933,0.002030,0.617253,0.014027
4,Berlin-Tiergarten,0.007209,0.002135,0.441725,0.013210
5,Eastern-Massachusetts,0.011850,0.002339,0.560020,0.018310
